Seguindo o curso de Pandas do canal do YouTube Téo Me Why, o tópico principal agora é o tratamento de dados faltantes e duplicatas. Pensando nisso, para separar bem o aprendizado de uma área mais hands-on e também devido a importância do tratamento no dia a dia de um analista de dados, eu prefiri criar esse outro notebook focado apenas nesse processo.

Dataset utilizado (Kaggle): https://www.kaggle.com/datasets/ahmedmohamed2003/cafe-sales-dirty-data-for-cleaning-training

Esta é apenas uma primeira prática geral de limpeza e tratamento de dados, então focarei em buscar valores nulos e duplicados e decidir a melhor forma de lidar com eles mantendo o dataset estruturado. O meu objetivo é manter no mínimo 85% do dataset ao final do processo, permitindo que análises posteriores não sejam influenciadas por uma grande remoção de dados.

In [1]:
import pandas as pd
import numpy as np

O primeiro passo é importar os dados e analisar a quantidade inicial de linhas totais e faltantes e as colunas

In [2]:
df = pd.read_csv('../data/kaggle/dirty_cafe_sales.csv')
df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [3]:
qtde_inicial = df.shape[0]
print(f'o dataset possui {df.shape[0]} linhas')
print('-'*40)
df.info(memory_usage='deep')

o dataset possui 10000 linhas
----------------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction ID    10000 non-null  str  
 1   Item              9667 non-null   str  
 2   Quantity          9862 non-null   str  
 3   Price Per Unit    9821 non-null   str  
 4   Total Spent       9827 non-null   str  
 5   Payment Method    7421 non-null   str  
 6   Location          6735 non-null   str  
 7   Transaction Date  9841 non-null   str  
dtypes: str(8)
memory usage: 4.1 MB


A primeira tentativa de limpar os dados nulos é retirar as linhas que possuem todos os valores faltantes. Ao fazer o comando, foi constatado que nenhuma possuía esse padrão.

In [4]:
df = df.dropna(how='all')
print(f'Restaram {df.shape[0]} linhas')

Restaram 10000 linhas


Com isso, decidi começar a analisar cada coluna do conjunto, nossas variáveis.

In [5]:
for variavel in df:
    print(variavel)
    print(f'a variável possui os {df[variavel].nunique()} valores únicos: ')
    print(df[variavel].unique())
    print()
    print('-'*40)

Transaction ID
a variável possui os 10000 valores únicos: 
<StringArray>
['TXN_1961373', 'TXN_4977031', 'TXN_4271903', 'TXN_7034554', 'TXN_3160411',
 'TXN_2602893', 'TXN_4433211', 'TXN_6699534', 'TXN_4717867', 'TXN_2064365',
 ...
 'TXN_1538510', 'TXN_3897619', 'TXN_2739140', 'TXN_4766549', 'TXN_7851634',
 'TXN_7672686', 'TXN_9659401', 'TXN_5255387', 'TXN_7695629', 'TXN_6170729']
Length: 10000, dtype: str

----------------------------------------
Item
a variável possui os 10 valores únicos: 
<StringArray>
[  'Coffee',     'Cake',   'Cookie',    'Salad', 'Smoothie',  'UNKNOWN',
 'Sandwich',        nan,    'ERROR',    'Juice',      'Tea']
Length: 11, dtype: str

----------------------------------------
Quantity
a variável possui os 7 valores únicos: 
<StringArray>
['2', '4', '5', '3', '1', 'ERROR', 'UNKNOWN', nan]
Length: 8, dtype: str

----------------------------------------
Price Per Unit
a variável possui os 8 valores únicos: 
<StringArray>
['2.0', '3.0', '1.0', '5.0', '4.0', '1.5', n

Identificando que as seguintes variáveis (Item, Quantity, Price Per Unity, Total Spent, Payment Method, Location) possuem os valores "ERROR" e "UNKNOWN", eu transformo esses valores para todas as variáveis em NaN para que eu possa trabalhar de melhor maneira

In [6]:
replace = {
    'ERROR' : np.nan,
    'UNKNOWN' : np.nan
}

df= df.replace(replace)
df

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,NaN,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,NaN,NaN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,2,2.0,4.0,NaN,NaN,2023-08-30
9996,TXN_9659401,NaN,3,NaN,3.0,Digital Wallet,NaN,2023-06-02
9997,TXN_5255387,Coffee,4,2.0,8.0,Digital Wallet,NaN,2023-03-02
9998,TXN_7695629,Cookie,3,NaN,3.0,Digital Wallet,NaN,2023-12-02


Antes de começar a pensar nas variáveis quantitativas em específico, eu decido remover as linhas do conjunto que possuem tanto o Item quanto o Preço por Unidade nulos, tendo em vista que o conjunto possui um cardápio pré-montado que não poderia ser utilizado caso ambos fossem nulos.

In [7]:
unidades = ['Item', 'Price Per Unit']
df = df.dropna(how='all', subset=unidades)
df

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,NaN,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,NaN,NaN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
...,...,...,...,...,...,...,...,...
9994,TXN_7851634,NaN,4,4.0,16.0,NaN,NaN,2023-01-08
9995,TXN_7672686,Coffee,2,2.0,4.0,NaN,NaN,2023-08-30
9997,TXN_5255387,Coffee,4,2.0,8.0,Digital Wallet,NaN,2023-03-02
9998,TXN_7695629,Cookie,3,NaN,3.0,Digital Wallet,NaN,2023-12-02


Agora que foram removidos os conflitos com o cardápio, eu peguei o dado desse cardápio disponível no kaggle e criei um dicionário para mapear e transformar os valores nulos de Item e Preço por Unidade em valores válidos.

Existe apenas um conflito, que é: Os preços de 'Sandwich' e 'Smoothie' são 4.0 e 'Cake' e 'Juice' são 3.0, logo, não como saber especificamente qual o item pedido, porém por convenção todos os item de 4.0 serão 'Sandwich' e todos os de 3.0 serão 'Cake'.

In [8]:
dict = {
    # Item -> Preço
    'Coffee' : '2.0',
    'Tea' : '1.5',
    'Sandwich' : '4.0',
    'Salad' : '5.0',
    'Cake' : '3.0',
    'Cookie' : '1.0',
    'Smoothie' : '4.0',
    'Juice' : '3.0',

    # Preço -> Item
    '2.0' : 'Coffee',
    '1.5' : 'Tea',
    '4.0' : 'Sandwich',
    '5.0' : 'Salad',
    '3.0' : 'Cake',
    '1.0' : 'Cookie'
}
df['Price Per Unit'] = df['Price Per Unit'].fillna(df['Item'].map(dict))
df['Item'] = df['Item'].fillna(df['Price Per Unit'].map(dict))
df

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,NaN,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,NaN,NaN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
...,...,...,...,...,...,...,...,...
9994,TXN_7851634,Sandwich,4,4.0,16.0,NaN,NaN,2023-01-08
9995,TXN_7672686,Coffee,2,2.0,4.0,NaN,NaN,2023-08-30
9997,TXN_5255387,Coffee,4,2.0,8.0,Digital Wallet,NaN,2023-03-02
9998,TXN_7695629,Cookie,3,1.0,3.0,Digital Wallet,NaN,2023-12-02


Agora, irei separar as variáveis qualitativas e quantitativas, desconsiderando as variáveis 'Transaction ID' e 'Transaction Date', devido a primeira não haver necessidade de tratamento e a segunda possuir uma tipagem especial que ocasionará em uma operação separada.

In [9]:
quantitativas = ['Quantity', 'Price Per Unit', 'Total Spent']
qualitativas = ['Item', 'Payment Method', 'Location']

Nesse primeiro momento, olharei para as variáveis quantitativas.

O primeiro passo é transformar elas para um tipo númerico, o float, para fazer operações;

O segundo passo é corrigir e aumentar o número de dados válidos da variável 'Total Spent', já que ela é o produto da 'Quantity' por 'Price Per Unit';

Além disso, também há a possibilidade de corrigir a variável 'Quantity', já que seu resultado é a divisão entre 'Total Spent' e 'Price Per Unit'

In [10]:
df[quantitativas] = df[quantitativas].astype(float)
df['Total Spent'] = df['Total Spent'].fillna(df['Quantity'] * df['Price Per Unit'])
df['Quantity'] = df['Quantity'].fillna(df['Total Spent'] / df['Price Per Unit'])
df.info()

<class 'pandas.DataFrame'>
Index: 9946 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction ID    9946 non-null   str    
 1   Item              9946 non-null   str    
 2   Quantity          9926 non-null   float64
 3   Price Per Unit    9946 non-null   float64
 4   Total Spent       9926 non-null   float64
 5   Payment Method    6789 non-null   str    
 6   Location          6010 non-null   str    
 7   Transaction Date  9489 non-null   str    
dtypes: float64(3), str(5)
memory usage: 699.3 KB


Agora, irei retirar os valores nulos ainda presentes em 'Quantity' e 'Total Spent', já que não há mais ação possível de validar esse dados.

Com isso, todas as variáveis quantitativas agora possuem o mesmo número de linhas, o que permite agora a análise das variáveis qualitativas.

In [11]:
df = df.dropna(how='any', subset=['Quantity', 'Total Spent'])
df.info()

<class 'pandas.DataFrame'>
Index: 9926 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction ID    9926 non-null   str    
 1   Item              9926 non-null   str    
 2   Quantity          9926 non-null   float64
 3   Price Per Unit    9926 non-null   float64
 4   Total Spent       9926 non-null   float64
 5   Payment Method    6776 non-null   str    
 6   Location          5996 non-null   str    
 7   Transaction Date  9469 non-null   str    
dtypes: float64(3), str(5)
memory usage: 697.9 KB


Como item também já está validado e com o mesmo número de dados que as variáveis quantitativas, restam analisar as variáveis Payment Method e Location.

Iniciando por Payment Method, eu decidi analisar se há alguma correlação entre os valores mais baixos (>= 12.5) e os valores mais altos  (< 12.5) com o metódo de pagamento escolhido. 

In [12]:
filtro_baixo = (df['Total Spent'] >= 1.0) & (df['Total Spent'] <= 12.5)

df[filtro_baixo]['Payment Method'].value_counts(normalize=True)

Payment Method
Digital Wallet    0.336613
Credit Card       0.333912
Cash              0.329475
Name: proportion, dtype: float64

In [13]:
filtro_alto = (df['Total Spent'] > 12.5) & (df['Total Spent'] <= 25.0) 
df[filtro_alto]['Payment Method'].value_counts(normalize=True)

Payment Method
Cash              0.337940
Digital Wallet    0.334171
Credit Card       0.327889
Name: proportion, dtype: float64

Como é possível observar, independente do valor gasto ser alto ou baixo, a frequência de uso dos métodos de pagamento é similar. A partir disso, há alguns métodos de resolução, como aleatorizar o método de pagamento mantendo a frequência de ~33% para cada (algo a ser incrementado no futuro), porém nesse momento irei apenas informar esses valores nulos como 'Não Informado'.

In [14]:
df[qualitativas] = df[qualitativas].fillna('Não Informado')
df

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4.0,1.0,4.0,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2.0,5.0,10.0,Não Informado,Não Informado,2023-04-27
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11
...,...,...,...,...,...,...,...,...
9994,TXN_7851634,Sandwich,4.0,4.0,16.0,Não Informado,Não Informado,2023-01-08
9995,TXN_7672686,Coffee,2.0,2.0,4.0,Não Informado,Não Informado,2023-08-30
9997,TXN_5255387,Coffee,4.0,2.0,8.0,Digital Wallet,Não Informado,2023-03-02
9998,TXN_7695629,Cookie,3.0,1.0,3.0,Digital Wallet,Não Informado,2023-12-02


Agora, resta apenas a variável Transaction Date.

O primeiro passo é retirar os valores nulos ainda presentes na variável, que então será convertida em DateTime para melhor manipulação posterior.

In [15]:
df = df.dropna(how='any', subset=['Transaction Date'])
df['Transaction Date'] = pd.to_datetime(df['Transaction Date'])
df.info()

<class 'pandas.DataFrame'>
Index: 9469 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    9469 non-null   str           
 1   Item              9469 non-null   str           
 2   Quantity          9469 non-null   float64       
 3   Price Per Unit    9469 non-null   float64       
 4   Total Spent       9469 non-null   float64       
 5   Payment Method    9469 non-null   str           
 6   Location          9469 non-null   str           
 7   Transaction Date  9469 non-null   datetime64[us]
dtypes: datetime64[us](1), float64(3), str(4)
memory usage: 665.8 KB


Com a variável Transaction Date convertida, o post do Kaggle de onde vem essa base de dados indica a criação de algumas variáveis relevantes baseadas na data, que são Day of The Week e Transaction Month, ambos para análises posteriores.

In [16]:
df['Day of the Week'] = df['Transaction Date'].dt.day_of_week
df

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date,Day of the Week
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08,4
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16,1
2,TXN_4271903,Cookie,4.0,1.0,4.0,Credit Card,In-store,2023-07-19,2
3,TXN_7034554,Salad,2.0,5.0,10.0,Não Informado,Não Informado,2023-04-27,3
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11,6
...,...,...,...,...,...,...,...,...,...
9994,TXN_7851634,Sandwich,4.0,4.0,16.0,Não Informado,Não Informado,2023-01-08,6
9995,TXN_7672686,Coffee,2.0,2.0,4.0,Não Informado,Não Informado,2023-08-30,2
9997,TXN_5255387,Coffee,4.0,2.0,8.0,Digital Wallet,Não Informado,2023-03-02,3
9998,TXN_7695629,Cookie,3.0,1.0,3.0,Digital Wallet,Não Informado,2023-12-02,5


In [17]:
df['Transaction Month'] = df['Transaction Date'].dt.month_name()
df

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date,Day of the Week,Transaction Month
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08,4,September
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16,1,May
2,TXN_4271903,Cookie,4.0,1.0,4.0,Credit Card,In-store,2023-07-19,2,July
3,TXN_7034554,Salad,2.0,5.0,10.0,Não Informado,Não Informado,2023-04-27,3,April
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11,6,June
...,...,...,...,...,...,...,...,...,...,...
9994,TXN_7851634,Sandwich,4.0,4.0,16.0,Não Informado,Não Informado,2023-01-08,6,January
9995,TXN_7672686,Coffee,2.0,2.0,4.0,Não Informado,Não Informado,2023-08-30,2,August
9997,TXN_5255387,Coffee,4.0,2.0,8.0,Digital Wallet,Não Informado,2023-03-02,3,March
9998,TXN_7695629,Cookie,3.0,1.0,3.0,Digital Wallet,Não Informado,2023-12-02,5,December


Com isso, o dataset agora está preparado para análises posteriores. Utilizando a quantidade inicial e a quantidade final do dataset, irei verificar a porcentagem de dados que mantive da base de dados original.

Após a verificação, 94.69% do dataset original foi mantido, o que está dentro do critério de perda máxima de 15% dos dados originais.

In [18]:
pct_final = (df.shape[0]/qtde_inicial) * 100
print(f'O dataset possui {pct_final}% do original')

O dataset possui 94.69% do original


Por fim, resta apenas exportar o dataset limpo para utilizá-lo em análises posteriores.

In [19]:
df.to_csv('cleaned_cafe_sales.csv', sep=',', index=False)
print('dataset salvo.')

dataset salvo.
